# Notebook 6 — Sparsity, robustness, and what the benchmark will not tell you

**Day 6.** The last one. Read this after Lecture 6, with `L1.prox`, `ProximalGradient`,
`Huber` and `PoissonNLL` written.

Five days of this course assumed the objective was differentiable everywhere. Today two
of the most useful objectives in statistics are not:

- $\|w\|_1$ has a kink at zero — and that kink is *the entire point*, because it is what
  sets coefficients to exactly zero rather than merely small;
- the Huber loss has a change of regime at $\pm\delta$, which is what stops a handful of
  outliers from dragging the fit across the room.

Then we do the thing the week has been building towards: run everything against
everything, and read the result honestly — including the part the benchmark cannot see.

1. what `prox` actually does, in one picture;
2. ISTA and FISTA, and the measurement that explains the gap between them;
3. lasso against ridge — exact zeros against merely small ones;
4. Huber against squared error, with the $\delta$ sweep that connects them;
5. the benchmark, and its most important limitation.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The optlab root is the nearest ancestor holding pyproject.toml, so this works whether
# Jupyter was started in notebooks/ or in the repository root.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), HERE.parent)

try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))
    import optlab

sys.path.insert(0, str(ROOT))  # for datasets/

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

In [ ]:
def status(name, thunk):
    """Report whether one piece of the package is implemented, without a traceback."""
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"

In [ ]:
from optlab.errors import LineSearchFailed, NotPositiveDefiniteError
from optlab.linalg import CholeskySolver
from optlab.linesearch import Armijo
from optlab.losses import Huber, LogisticNLL, PoissonNLL, SquaredError
from optlab.objective_ops import RegularizedObjective
from optlab.observers import History
from optlab.optimizers import (DescentOptimizer, HeavyBall, NewtonDirection,
                               ProximalGradient, SteepestDescent, newton)
from optlab.problems import GLMLoss, Quadratic, Rosenbrock, logistic_regression
from optlab.regularizers import ElasticNet, L1, L2, NoRegularizer
from optlab.stopping import AnyOf, GradientNormBelow, MaxIterations

_v = np.array([-1.0, 0.0, 1.0])
_glm = GLMLoss(np.eye(2), np.array([1.0, 0.0]), SquaredError())

print("Day 6 readiness")
print(status("L1.prox", lambda: L1(0.5).prox(_v, 1.0)))
print(status("L1.value", lambda: L1(0.5).value(_v)))
print(status("ElasticNet.prox", lambda: ElasticNet(0.5, 0.5).prox(_v, 1.0)))
print(status("Huber", lambda: Huber(1.0).value(0.5, 0.0)))
print(status("PoissonNLL", lambda: PoissonNLL().value(0.5, 1.0)))
print(status("ProximalGradient",
             lambda: ProximalGradient(_glm, L1(0.1), step=0.1,
                                      max_iter=2).minimize(None, np.zeros(2))))
print()
print("L1.gradient is SUPPOSED to raise - that is the contract, not a gap:")
try:
    L1(1.0).gradient(_v)
    print("  !! it did not raise. Re-read the stub docstring.")
except NotImplementedError as err:
    print(f"  correct: NotImplementedError({err})")
print()
print("From earlier days")
print(status("L2.prox", lambda: L2(0.5).prox(_v, 1.0)))
print(status("CholeskySolver", lambda: CholeskySolver().solve(np.eye(2), np.ones(2))))
print(status("newton", lambda: newton(max_iter=2).minimize(_glm, np.zeros(2))))

## 1. The proximal operator, in one picture

Gradient descent cannot minimise $\|w\|_1$, because at the place where the solution wants
to sit there is no gradient. The proximal operator is the replacement:

$$\operatorname{prox}_{t\,r}(v) \;=\; \arg\min_u \Bigl\{ r(u) + \tfrac{1}{2t}\|u - v\|^2 \Bigr\}.$$

Read it as "move towards minimising $r$, but do not go far from $v$". For our three
penalties it has a closed form, and the three forms are visibly different animals:

| penalty | prox | what it does |
|---|---|---|
| $\tfrac{1}{2}\lambda\|w\|^2$ (ridge) | $v / (1 + \lambda t)$ | scales everything down |
| $\lambda\|w\|_1$ (lasso) | $\operatorname{sign}(v)\max(|v| - \lambda t,\, 0)$ | **subtracts** a constant, clipping at 0 |
| elastic net | soft-threshold, then scale | both |

The difference between *scaling* and *subtracting* is the whole of this section.

In [ ]:
v = np.linspace(-2.0, 2.0, 401)
lam, t = 0.5, 1.0

prox_l1 = L1(lam).prox(v, t)
prox_l2 = L2(lam).prox(v, t)
prox_en = ElasticNet(lam, 0.5).prox(v, t)

probe = np.array([-2.0, -0.6, -0.4, -0.1, 0.0, 0.1, 0.4, 0.6, 2.0])
print(f"{'v':>8} {'L1 prox':>10} {'L2 prox':>10} {'EN prox':>10}")
for a, p1, p2, pe in zip(probe, L1(lam).prox(probe, t), L2(lam).prox(probe, t),
                         ElasticNet(lam, 0.5).prox(probe, t)):
    print(f"{a:8.2f} {p1:10.4f} {p2:10.4f} {pe:10.4f}")

# Count only the probe points that were NOT already zero - v = 0 stays 0 under any
# prox, so including it would flatter both methods equally and prove nothing.
nonzero_in = probe != 0.0
made_zero_l1 = np.sum(L1(lam).prox(probe, t)[nonzero_in] == 0.0)
made_zero_l2 = np.sum(L2(lam).prox(probe, t)[nonzero_in] == 0.0)
print(f"\nof the {nonzero_in.sum()} probe points that were not already zero,")
print(f"  L1 sent {made_zero_l1} of them to exactly 0.0")
print(f"  L2 sent {made_zero_l2} of them to exactly 0.0")
print(f"  smallest |L2 prox| among those: "
      f"{np.min(np.abs(L2(lam).prox(probe, t)[nonzero_in])):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

ax[0].plot(v, v, "k:", lw=1, label="identity (no penalty)")
ax[0].plot(v, prox_l2, label=r"L2: $v/(1+\lambda t)$")
ax[0].plot(v, prox_en, label=r"ElasticNet($\lambda$, 0.5)")
ax[0].plot(v, prox_l1, lw=2, label=r"L1: soft-threshold")
ax[0].axvspan(-lam * t, lam * t, color="tab:red", alpha=0.10)
ax[0].annotate("everything in here\nis sent to exactly 0", (0, -1.2), ha="center",
               fontsize=7.5, color="tab:red")
ax[0].set_xlabel("$v$")
ax[0].set_ylabel(r"$\mathrm{prox}_{t r}(v)$")
ax[0].set_title(r"The three prox operators, $\lambda = 0.5$, $t = 1$")
ax[0].legend(fontsize=8)

near = np.linspace(-0.05, 0.05, 401)
ax[1].plot(near, near, "k:", lw=1, label="identity")
ax[1].plot(near, L2(lam).prox(near, t), label="L2")
ax[1].plot(near, L1(lam).prox(near, t), lw=2, label="L1")
ax[1].set_xlabel("$v$")
ax[1].set_title("Zoomed on the origin: L2 never reaches zero,\nL1 is already there")
ax[1].legend(fontsize=8)

fig.suptitle("Figure 1 — why L1 produces zeros and L2 does not")
fig.tight_layout()
plt.show()

**Figure 1.** The L2 prox is a line through the origin with slope $1/(1+\lambda t)$: it
multiplies. A number that is small becomes smaller, and never becomes zero — the zoomed
panel on the right shows it still descending through $v = \pm 0.05$, and it would still be
descending at $10^{-300}$.

The L1 prox is flat at exactly zero on the whole shaded band $|v| \le \lambda t$, then
parallel to the identity outside it: it *subtracts*. Of the eight probe values that were
not already zero, L1 sent **4** to exactly `0.0` — precisely the four with
$|v| \le \lambda t = 0.5$ — and L2 sent **0**. The smallest magnitude L2 produced was
$0.0667$, from an input of $0.1$. "Exactly" here means the float is `0.0`, not $10^{-12}$;
you can test it with `==`.

> **`-0.0000` in the table is not a bug.** `sign(v) * max(|v| - λt, 0)` for negative $v$
> gives $-1 \times 0.0 = -0.0$, IEEE negative zero. It compares equal to `0.0`, prints
> with a minus sign, and is genuinely zero. Do not "fix" it by adding `+ 0.0`.

That distinction is the difference between a model with 400 tiny coefficients and a model
with 5. Only the second one can be read.

> **Why `L1.gradient` raises.** The readiness cell above confirmed it. L1 is not
> differentiable at zero, so there is no gradient to return there, and returning
> $\operatorname{sign}(0) = 0$ would be a lie that lets gradient descent sail straight
> through the kink — destroying the very property you wanted. `ProximalGradient` never
> calls `gradient` on the penalty. The raise documents the design; it is not a gap.

## 2. ISTA and FISTA

The proximal gradient method (ISTA) alternates: one gradient step on the smooth part, one
prox of the penalty.

$$w_{k+1} = \operatorname{prox}_{\alpha r}\bigl(w_k - \alpha \nabla f(w_k)\bigr),
\qquad \alpha \le 1/L .$$

FISTA adds Nesterov extrapolation, taking the step from a point slightly *beyond* the
last iterate. The textbook claim is that this improves the bound from $O(1/k)$ to
$O(1/k^2)$. We will measure it on a real lasso: $n = 100$ samples, $d = 400$ features,
5 of which are genuinely non-zero. Note $d > n$ — that is the regime lasso exists for,
and it also means the smooth part is **not** strongly convex, which is exactly when the
sublinear rates apply.

In [ ]:
n, d = 100, 400
lasso_rng = np.random.default_rng(7)
X = lasso_rng.normal(size=(n, d))
w_true = np.zeros(d)
true_support = [0, 3, 7, 20, 150]
w_true[true_support] = [3.0, -2.0, 1.5, -1.0, 2.5]
y = X @ w_true + lasso_rng.normal(scale=0.3, size=n)

smooth = GLMLoss(X, y, SquaredError())
lam = 0.05
eigs = np.linalg.eigvalsh(X.T @ X / n)
L_smooth = eigs[-1]
step = 1.0 / L_smooth
print(f"n = {n}, d = {d}, true non-zeros = {len(true_support)}")
print(f"L = {L_smooth:.4f}, step = 1/L = {step:.4f}")
print(f"smallest Hessian eigenvalue = {eigs[0]:.2e}  (zero: not strongly convex)")

full_objective = lambda w: smooth.value(w) + lam * np.sum(np.abs(w))

# A high-accuracy reference minimum. tol=-1.0 disables the early return so the run
# uses its whole budget.
ref = ProximalGradient(smooth, L1(lam), step=step, max_iter=100000, tol=-1.0,
                       accelerated=True).minimize(None, np.zeros(d))
f_star = full_objective(ref.x)
print(f"f* = {f_star:.12f}   non-zeros at the optimum = {np.sum(np.abs(ref.x) > 1e-8)}")

curves = {}
for accel, name in ((False, "ISTA"), (True, "FISTA")):
    hist = History()
    ProximalGradient(smooth, L1(lam), step=step, max_iter=500, tol=-1.0,
                     accelerated=accel, observers=[hist]).minimize(None, np.zeros(d))
    gaps = np.array(hist.values) - f_star
    nnz = np.array([np.sum(np.abs(e.x) > 1e-8) for e in hist.events])
    curves[name] = (gaps, nnz)
    k = np.arange(1, gaps.size + 1)
    window = (k >= 5) & (k <= 50) & (gaps > 1e-13)
    slope = np.polyfit(np.log(k[window]), np.log(gaps[window]), 1)[0]
    settled = np.argmax(nnz == nnz[-1]) + 1
    reach = lambda tgt: (np.argmax(gaps <= tgt) + 1) if (gaps <= tgt).any() else None
    print(f"\n{name}:  log-log slope over k=5..50 : {slope:.2f}")
    print(f"  iterations to gap 1e-6 / 1e-9 : {reach(1e-6)} / {reach(1e-9)}")
    print(f"  non-zeros after 1 iteration   : {nnz[0]} of {d}")
    print(f"  support size settles at {nnz[-1]} from iteration {settled}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))

k = np.arange(1, curves["ISTA"][0].size + 1)
for name, style in (("ISTA", "-"), ("FISTA", "-")):
    gaps = np.maximum(curves[name][0], 1e-16)
    ax[0].loglog(k, gaps, style, lw=1.4, label=name)
anchor = curves["ISTA"][0][9]
ax[0].loglog(k, anchor * (10.0 / k), "k--", lw=1, label=r"slope $-1$ (the $O(1/k)$ rate)")
ax[0].loglog(k, anchor * (10.0 / k) ** 2, "k:", lw=1,
             label=r"slope $-2$ (the $O(1/k^2)$ rate)")
ax[0].set_ylim(1e-16, 1e2)
ax[0].set_xlabel("iteration $k$")
ax[0].set_ylabel(r"$F(w_k) - F^*$")
ax[0].set_title("Both beat their own bound")
ax[0].legend(fontsize=8)

for name in ("ISTA", "FISTA"):
    ax[1].plot(k, curves[name][1], lw=1.4, label=name)
ax[1].axhline(len(true_support), color="k", ls="--", lw=1,
              label=f"{len(true_support)} truly non-zero")
ax[1].axhline(curves["ISTA"][1][-1], color="tab:red", ls=":", lw=1,
              label=f"{curves['ISTA'][1][-1]} at the optimum")
ax[1].set_xlim(0, 350)
ax[1].set_xlabel("iteration $k$")
ax[1].set_ylabel("number of non-zero coefficients")
ax[1].set_title("Finding the support is what takes the time")
ax[1].legend(fontsize=8)

fig.suptitle(r"Figure 2 — lasso, $n=100$, $d=400$, $\lambda = 0.05$")
fig.tight_layout()
plt.show()

**Figure 2, left.** The two black reference lines have slope $-1$ and $-2$, the rates the
theory quotes; their vertical position is arbitrary (both are pinned to ISTA at $k = 10$),
so compare *slopes*, not heights. Over $k = 5 \dots 50$ the measured log-log slopes are
$-0.36$ for ISTA and $-2.79$ for FISTA.

Neither matches. ISTA is decaying more slowly than $-1$ in this window and FISTA is
decaying faster than $-2$ — which is your reminder that $O(1/k)$ and $O(1/k^2)$ are
worst-case guarantees over an entire function class, not predictions about any one run.
And then both curves fall off a cliff and reach $10^{-13}$, which no sublinear rate
predicts at all.

**Figure 2, right, explains all of it.** Both methods start with almost every coefficient
non-zero — the printed count after one iteration is 368 of 400 — and grind downward. ISTA
reaches the final support of 25 at iteration 255; FISTA gets there at iteration 59,
**more than four times sooner**.

That is the real mechanism. While the support is wrong the method is searching a
400-dimensional problem. Once the support is correct, the problem has collapsed to 25
dimensions *and is strongly convex there*, so convergence turns linear and finishes almost
at once. Proximal methods identify the active set in finitely many steps, and after that
they are solving a much smaller problem.

So FISTA's advantage here is not really the $O(1/k^2)$ of the textbook — it is that
momentum finds the right 25 coefficients four times faster. The iteration counts to a gap
of $10^{-6}$ are 297 for ISTA and 118 for FISTA, a ratio of $2.5$; a figure you would
never have predicted from the ratio of the two bounds.

## 3. Lasso against ridge: exact zeros against merely small ones

Notebook 4 §5 built a ridge path and noted that a coefficient whose true value was
exactly $0$ came back as $0.81$ — ridge shrinks but does not select. Here is the
comparison done properly, on a problem where only 5 of 400 coefficients are real.

Both paths use the *same optimizer*. The only thing that changes is which `IRegularizer`
object gets passed in, which is the design point of day 4 and day 6 together.

In [ ]:
lambdas = np.logspace(-0.3, -2.3, 14)
lasso_path, ridge_path = [], []

print(f"{'lambda':>9} {'lasso non-zero':>15} {'ridge non-zero':>15} "
      f"{'true 5 found?':>14} {'extra':>7}")
for lm in lambdas:
    r_l1 = ProximalGradient(smooth, L1(lm), step=step, max_iter=5000, tol=1e-12,
                            accelerated=True).minimize(None, np.zeros(d))
    r_l2 = ProximalGradient(smooth, L2(lm), step=step, max_iter=5000, tol=1e-12,
                            accelerated=True).minimize(None, np.zeros(d))
    lasso_path.append(r_l1.x)
    ridge_path.append(r_l2.x)
    support = set(np.flatnonzero(np.abs(r_l1.x) > 1e-8))
    n_l2 = int(np.sum(np.abs(r_l2.x) > 1e-8))
    found = set(true_support) <= support
    print(f"{lm:9.4f} {len(support):15d} {n_l2:15d} {str(found):>14} "
          f"{len(support - set(true_support)):7d}")

lasso_path = np.array(lasso_path)
ridge_path = np.array(ridge_path)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

for j in range(d):
    is_true = j in true_support
    ax[0].semilogx(lambdas, lasso_path[:, j],
                   color="tab:red" if is_true else "0.8",
                   lw=1.6 if is_true else 0.5, zorder=3 if is_true else 1)
    ax[1].semilogx(lambdas, ridge_path[:, j],
                   color="tab:red" if is_true else "0.8",
                   lw=1.6 if is_true else 0.5, zorder=3 if is_true else 1)
for a, title in ((ax[0], "Lasso: the grey lines are flat at 0"),
                 (ax[1], "Ridge: the grey lines are near 0, never 0")):
    a.axhline(0.0, color="k", lw=0.8)
    a.invert_xaxis()
    a.set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
    a.set_ylabel("coefficient")
    a.set_title(title)
ax[0].plot([], [], color="tab:red", lw=1.6, label="truly non-zero (5)")
ax[0].plot([], [], color="0.8", lw=1.0, label="truly zero (395)")
ax[0].legend(fontsize=7.5)

ax[2].semilogx(lambdas, (np.abs(lasso_path) > 1e-8).sum(axis=1), "o-", label="lasso")
ax[2].semilogx(lambdas, (np.abs(ridge_path) > 1e-8).sum(axis=1), "s-", label="ridge")
ax[2].axhline(len(true_support), color="k", ls="--", lw=1, label="the truth: 5")
ax[2].set_yscale("log")
ax[2].invert_xaxis()
ax[2].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[2].set_ylabel("coefficients that are not exactly 0")
ax[2].set_title("Model size against $\\lambda$")
ax[2].legend(fontsize=8)

fig.suptitle("Figure 3 — 400 features, 5 of them real")
fig.tight_layout()
plt.show()

**Figure 3.** The right panel is the one to remember. Ridge sits flat on **400** at every
single $\lambda$: it never sets anything to zero, so every model on the ridge path uses
every feature. Lasso starts at 5 and grows as $\lambda$ falls.

And look at the table. For the top four $\lambda$ values — $0.50$ down to $0.17$ — the
lasso support is **exactly the 5 true coefficients**: `True` in the "found" column and `0`
in the "extra" column. From 400 candidates, with only 100 samples, it recovered precisely
the right five. That is not shrinkage, it is model selection, and it is what people
actually use lasso for.

Read the failure mode too. The "found" column stays `True` at every single $\lambda$ —
lasso never *drops* a real coefficient here — but the "extra" column grows without
stopping: 0, 0, 0, 0, then 1, 5, 12, 25, 38, 51, 61, 71, 79, 81. Every one of those is a
false positive. So the two kinds of error are not symmetric on this path: lowering
$\lambda$ costs you precision, not recall.

Lasso does not hand you the truth; it hands you a *path*. Choosing where to stand on it is
a statistical decision — cross-validation, usually — and no optimizer will make it for
you.

In the left panel the five red lines climb away from zero as $\lambda$ falls while the 395
grey lines lie flat on the axis, peeling off one by one. In the middle panel the same 395
grey lines are all slightly off the axis at every $\lambda$, which is the picture of
shrinking without selecting.

## 4. Huber: what a squared error costs you when the data lies

Squared error is the Gaussian negative log-likelihood (notebook 1 §1), and the Gaussian
has very thin tails. It assigns an observation 12 units away a log-likelihood so bad that
the fit will move a long way to reduce it. If that observation is a transcription error,
the fit has been moved by a typo.

Huber is quadratic within $\pm\delta$ and linear beyond:

$$\varphi_\delta(z, y) = \begin{cases}
\tfrac{1}{2}(z-y)^2 & |z - y| \le \delta \\
\delta\,|z-y| - \tfrac{1}{2}\delta^2 & \text{otherwise.}
\end{cases}$$

Linear tails mean a far-away point exerts a *bounded* pull, however far away it goes. We
fit a straight line and corrupt points only on the right-hand third of the range, which
is the dangerous case: one-sided errors tilt the slope rather than just shifting the
intercept.

In [ ]:
m = 120
t_line = np.linspace(-2.0, 2.0, m)
X_line = np.column_stack([np.ones(m), t_line])
beta_true = np.array([1.0, 2.0])                 # intercept, slope
out_rng = np.random.default_rng(31)
y_clean = X_line @ beta_true + out_rng.normal(scale=0.2, size=m)

right_side = np.flatnonzero(t_line > 0.7)
fits = {}

print(f"true (intercept, slope) = {beta_true}")
print(f"\n{'outliers':>9} {'squared error':>24} {'Huber(1.0)':>24}")
for n_out in (0, 4, 10, 20):
    y_bad = y_clean.copy()
    if n_out:
        idx = np.random.default_rng(100 + n_out).choice(right_side, n_out, replace=False)
        y_bad[idx] -= 12.0
    r_sq = newton(tol=1e-10, max_iter=300).minimize(
        GLMLoss(X_line, y_bad, SquaredError()), np.zeros(2))
    r_hb = newton(tol=1e-10, max_iter=600).minimize(
        GLMLoss(X_line, y_bad, Huber(1.0)), np.zeros(2))
    fits[n_out] = (y_bad, r_sq.x, r_hb.x)
    print(f"{n_out:9d}   ({r_sq.x[0]:8.4f}, {r_sq.x[1]:8.4f})     "
          f"({r_hb.x[0]:8.4f}, {r_hb.x[1]:8.4f})")

In [ ]:
y_bad, _, _ = fits[10]
deltas = [0.1, 0.3, 1.0, 3.0, 10.0, 100.0]
sweep = []
print("delta sweep on the 10-outlier data set")
for delta in deltas:
    r = newton(tol=1e-10, max_iter=900).minimize(
        GLMLoss(X_line, y_bad, Huber(delta)), np.zeros(2))
    sweep.append(r.x)
    print(f"  delta {delta:7.1f}: intercept {r.x[0]:8.4f}  slope {r.x[1]:8.4f}")
sweep = np.array(sweep)
sq_fit = fits[10][1]
print(f"  squared error : intercept {sq_fit[0]:8.4f}  slope {sq_fit[1]:8.4f}")
print(f"\n|Huber(delta=100) - squared error| = {np.max(np.abs(sweep[-1] - sq_fit)):.2e}")
print(f"largest |residual| at that fit     = {np.max(np.abs(X_line @ sq_fit - y_bad)):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))

for n_out, a in zip((0, 20), ax[:2]):
    y_bad_n, w_sq, w_hb = fits[n_out]
    a.plot(t_line, y_bad_n, "k.", ms=4)
    a.plot(t_line, X_line @ beta_true, "k--", lw=1, label="truth")
    a.plot(t_line, X_line @ w_sq, lw=2,
           label=f"squared: slope {w_sq[1]:.3f}")
    a.plot(t_line, X_line @ w_hb, lw=2,
           label=f"Huber:   slope {w_hb[1]:.3f}")
    a.set_xlabel("$t$")
    a.set_title(f"{n_out} corrupted points (all on the right)")
    a.legend(fontsize=8, loc="upper left")

ax[2].semilogx(deltas, sweep[:, 1], "o-", label="Huber slope")
ax[2].axhline(beta_true[1], color="k", ls="--", lw=1, label=f"truth {beta_true[1]}")
ax[2].axhline(sq_fit[1], color="tab:red", ls=":", lw=1.5,
              label=f"squared error {sq_fit[1]:.3f}")
ax[2].set_xlabel(r"Huber $\delta$")
ax[2].set_ylabel("fitted slope")
ax[2].set_title(r"$\delta \to \infty$ recovers squared error exactly")
ax[2].legend(fontsize=8)

fig.suptitle("Figure 4 — one-sided outliers, and the one parameter that controls the cure")
fig.tight_layout()
plt.show()

**Figure 4, left and middle.** With clean data the two losses give the **same** answer to
four decimals — $(0.990, 2.005)$ both. Huber costs nothing when there is nothing to be
robust against, which is the property that makes it usable as a default.

With 20 corrupted points out of 120 — one sixth of the data — squared error returns a
slope of $-0.065$. The true slope is $2.0$. The fit is not degraded, it is *destroyed*:
the blue line in the middle panel is flat, and someone reading it would conclude the
predictor has no effect at all, which is the opposite of the truth. Huber returns $1.703$
— not exact, but it still tells the right story, and the orange line still points the
right way.

**Figure 4, right.** The $\delta$ sweep is the part worth understanding, because it shows
the two losses are not rivals but two ends of one family. At $\delta = 0.1$ nearly every
residual is in the linear regime and the slope is $1.991$, almost exact. As $\delta$
grows, more residuals fall inside the quadratic region, the outliers regain their
influence, and the curve collapses onto the red squared-error line.

By $\delta = 100$ the difference printed above is `0.00e+00` — not small, **exactly zero**,
in every bit of both parameters. And the line below it says why: the largest residual at
that fit is $10.32$. No residual anywhere in the run ever reaches $\pm 100$, so
every point stays in the quadratic branch, where Huber's value, gradient and Hessian are
*identical expressions* to squared error's. Newton therefore takes bit-identical steps.
This is not an analogy or a limit in the loose sense: for this data, Huber with
$\delta = 100$ **is** squared error.

So $\delta$ is not a robustness knob to be tuned by taste. It is the residual size at
which you stop believing the Gaussian model. Set it from what you know about your
measurement error, and check it against the residuals you get.

## 5. The benchmark, and what it cannot see

Everything you have written this week, against a spread of problems. Same stopping rule,
same line search, same budget, and we record both iterations and wall-clock time —
because notebook 4 §4 showed those are different questions.

In [ ]:
import time


def spd_quadratic(kappa, dim=20, seed=0):
    """A quadratic with exactly the condition number asked for."""
    spectrum = np.logspace(0.0, np.log10(kappa), dim)
    Q, _ = np.linalg.qr(np.random.default_rng(seed).normal(size=(dim, dim)))
    return Quadratic(Q @ np.diag(spectrum) @ Q.T,
                     np.random.default_rng(seed + 1).normal(size=dim))


def logistic(n_obs, dim, spread=1.0, seed=11):
    gen = np.random.default_rng(seed)
    Xl = gen.normal(size=(n_obs, dim)) * np.linspace(1.0, spread, dim)
    yl = (gen.random(n_obs) < 1.0 / (1.0 + np.exp(-(Xl @ gen.normal(size=dim))))).astype(float)
    return logistic_regression(Xl, yl)


problems = {
    "quad k=10": (spd_quadratic(10), np.ones(20)),
    "quad k=1e3": (spd_quadratic(1e3), np.ones(20)),
    "quad k=1e5": (spd_quadratic(1e5), np.ones(20)),
    "rosenbrock": (Rosenbrock(), np.array([-1.2, 1.0])),
    "logistic easy": (logistic(400, 8), np.zeros(8)),
    "logistic d=150": (logistic(2000, 150), np.zeros(150)),
    "logistic scaled": (logistic(400, 8, spread=100.0), np.zeros(8)),
    "ridge logistic": (RegularizedObjective(logistic(200, 30, seed=5), L2(0.01)), np.zeros(30)),
}
methods = {
    "GD + Armijo": lambda: SteepestDescent(),
    "HeavyBall": lambda: HeavyBall(0.9),
    "Newton": lambda: NewtonDirection(CholeskySolver()),
}
BUDGET = 20000

iters = np.full((len(problems), len(methods)), np.inf)
secs = np.full((len(problems), len(methods)), np.inf)

print(f"{'problem':>16} " + "".join(f"{m:>24}" for m in methods))
for i, (pname, (prob, x0)) in enumerate(problems.items()):
    cells = []
    for j, (mname, make) in enumerate(methods.items()):
        try:
            t0 = time.perf_counter()
            r = DescentOptimizer(make(), Armijo(),
                                 AnyOf(GradientNormBelow(1e-6), MaxIterations(BUDGET))
                                 ).minimize(prob, x0)
            elapsed = time.perf_counter() - t0
            if r.converged:
                iters[i, j], secs[i, j] = r.iterations, elapsed
                cells.append(f"{r.iterations:7d} it {elapsed * 1e3:8.1f} ms")
            else:
                cells.append(f"{'did not converge':>21}")
        except (NotPositiveDefiniteError, LineSearchFailed) as err:
            cells.append(f"{type(err).__name__:>21}")
    print(f"{pname:>16} " + "".join(f"{c:>24}" for c in cells))

best_time = np.min(secs, axis=1)
print(f"\n{'method':>12} {'solved':>8} {'closest it ever got to the best':>34}")
for j, mname in enumerate(methods):
    ratio = secs[:, j] / best_time
    finite = ratio[np.isfinite(ratio)]
    closest = f"{finite.min():.1f}x" if finite.size else "n/a"
    print(f"  {mname:>10} {int(np.sum(np.isfinite(secs[:, j]))):8d} {closest:>34}")
print("\nTimings are machine-dependent - read the pattern, not the digits.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
names = list(problems)
mnames = list(methods)
xpos = np.arange(len(names))
width = 0.26

for j, mname in enumerate(mnames):
    col = secs[:, j] * 1e3
    drawn = np.where(np.isfinite(col), col, 1e-3)
    bars = ax[0].bar(xpos + (j - 1) * width, drawn, width, label=mname)
    for i, bar in enumerate(bars):
        if not np.isfinite(col[i]):
            ax[0].annotate("fail", (bar.get_x() + bar.get_width() / 2, 1.2e-3),
                           ha="center", fontsize=6.5, rotation=90, color="0.35")
ax[0].set_yscale("log")
ax[0].set_xticks(xpos)
ax[0].set_xticklabels(names, rotation=35, ha="right", fontsize=7.5)
ax[0].set_ylabel("time to $\\|\\nabla f\\| \\leq 10^{-6}$ (ms)")
ax[0].set_title("Wall-clock time (missing bar = did not solve it)")
ax[0].legend(fontsize=8)

# Performance profile: fraction of problems solved within a factor tau of the best.
best = np.nanmin(np.where(np.isfinite(secs), secs, np.nan), axis=1)
taus = np.logspace(0, 4, 200)
for j, mname in enumerate(mnames):
    ratio = np.where(np.isfinite(secs[:, j]), secs[:, j] / best, np.inf)
    frac = [(ratio <= tau).mean() for tau in taus]
    ax[1].semilogx(taus, frac, lw=1.8, label=mname)
ax[1].set_xlabel(r"$\tau$ = how many times slower than the best method")
ax[1].set_ylabel("fraction of problems solved")
ax[1].set_ylim(-0.03, 1.03)
ax[1].set_title("Performance profile (higher and further left is better)")
ax[1].legend(fontsize=8)

fig.suptitle("Figure 5 — everything from this week, on eight problems")
fig.tight_layout()
plt.show()

**Figure 5.** Newton solves 8 of 8 and is the fastest on every one: its curve in the
performance profile starts at $1.0$ and stays there, which is what total dominance looks
like on this kind of plot. Gradient descent solves 5, HeavyBall 3.

The profile says something sharper than the bar chart. Both first-order curves sit at $0$
for a long stretch before they rise at all: the "closest it ever got" column prints, for
each method, its *best* showing across all eight problems, and neither first-order method
manages to come within roughly an order of magnitude of Newton on even one of them. On the
problems they do solve the penalty runs into the hundreds and beyond.

(Your exact milliseconds will differ from the ones printed here — that is a statement
about this laptop, not about the algorithms. Read the pattern.)

Nor is this a lucky problem set. The three quadratics walk the condition number from $10$
to $10^5$ and Newton takes **1 iteration** on all three — it is affine invariant, so
conditioning simply does not reach it, while GD needs 54, then 6452, then fails.
`logistic d=150` is the one row where Newton is visibly slower in wall-clock (16 ms, its
worst), because forming and factorising a $150 \times 150$ Hessian costs real time — and
both first-order methods still fail outright on it.

Within this class of problems — dense, twice differentiable, $n$ moderate, $d$ in the
hundreds — the conclusion is simply **use Newton**. The rest of the week's methods are
what you reach for when one of those conditions fails.

**Now the important part, which is what the benchmark left out.**

Look at what is *not* in the table. Every problem here has a Hessian that can be formed
and factorised. Every method here is deterministic and uses a full gradient. And every row
is scored by "iterations to a gradient tolerance". Those three choices, taken together,
exclude by construction:

- **everything from day 3.** `SGD` and `Adam` cannot even be entered — they do not take a
  `IStoppingCriterion`, they count epochs, and they stop at a noise floor rather than at a
  tolerance. A harness scored on "iterations to $\|\nabla f\| \le 10^{-6}$" cannot
  represent a method that never reaches $10^{-6}$ and is not trying to.
- **the regime where that matters.** Every problem above has $n$ in the hundreds or low
  thousands, so one Hessian costs nothing. At $n = 10^8$ Newton cannot take its *first*
  step, because forming $X^\top D X$ requires a pass over data that does not fit in
  memory — while SGD has already made a thousand updates from mini-batches of 32. The
  benchmark contains no problem large enough for that to show up, so the crossover is
  simply absent from the plot.
- **day 5 and day 6 entirely.** `GaussNewton`, `LevenbergMarquardt` and
  `ProximalGradient` take different inputs (an `ILeastSquaresProblem`; a smooth part plus a
  penalty), so they cannot be plugged into this harness either. Lasso has no gradient at
  its own solution; the stopping test would never fire.

So the honest summary of Figure 5 is: *among the methods that fit this harness, on the
problems this harness can express, Newton wins.* Both halves of that sentence are doing
work. **A benchmark's most important limitation is almost always what it could not
include, and that limitation is invisible in the output** — there is no bar for the
missing methods, no column for the missing regime. You have to go and look at the harness.

That is the last idea of the week, and it is the one that outlives the code: measure, but
know exactly what you measured.

## Checkpoint

1. Someone reports that their lasso "shrinks but never zeroes anything". Using Figure 1,
   name the single most likely bug in their `prox`.
2. In Figure 2 FISTA reached the correct support at iteration 59 and ISTA at 255. Predict
   what happens to both numbers if you raise $\lambda$ to $0.5$, then check it.
3. Your lasso at the cross-validated $\lambda$ keeps 40 features and you know only 5 are
   real. Is that a bug? What does §3's table say, and what would you do about it?
4. You set Huber's $\delta$ to the standard deviation of your residuals and the fit barely
   changes from least squares. Using Figure 4's right panel, is $\delta$ too large or too
   small, and which direction should you move it?
5. Extend Figure 5 honestly: add a problem to `problems` on which Newton is *not* the
   best choice. You may not change the methods or the stopping rule — only the problem.
   (Hint: the constraint that matters is the one that stops you forming $H$.)

---

## That is the week

You built, from an empty package: losses derived from likelihoods, gradient descent with
a line search, three stochastic optimizers, Newton with a Cholesky solver that refuses
indefinite matrices, Gauss–Newton and Levenberg–Marquardt, and a proximal method for
objectives that are not differentiable. Roughly 20 classes, each one small, and the
interfaces held.

The habit those six days were really for is the one in the notebooks README:

> **derive it, code it, check it against an oracle, then break it.**

Every notebook this week ended by breaking something: a gradient check that passes on a
wrong gradient, a step size at exactly $2/L$, a noise floor that will not go away, a
Hessian Cholesky refuses, an LM that rejects a correct answer forever, and now a benchmark
whose real conclusion is about what it left out. Those are the sections that will still be
useful to you when the specific algorithms have gone.

A method you have only seen succeed is a method you do not yet understand.